In [ ]:
%run ./globalvariables

In [ ]:
import re
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StringType, IntegerType, LongType, ShortType, ByteType,
    DoubleType, FloatType, BooleanType, DateType, TimestampType, DecimalType
)

In [ ]:
def add_ingestion_metadata(df, source_system, source_file, partition_key):
    return (
        df.withColumn("ingestion_ts", current_timestamp())
          .withColumn("source_system", lit(source_system))
          .withColumn("source_file", lit(source_file))
          .withColumn("partition_key", lit(partition_key))
    )

In [ ]:
def fetch_with_retry(fn, attempts=3, base_delay=2):
    """Retry fn with exponential backoff on network failure."""
    for attempt in range(attempts):
        try:
            return fn()
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2**attempt)


def download_to_volume(url, dest_path):
    """Download a URL into a UC Volume and return the local path."""
    Path(dest_path).parent.mkdir(parents=True, exist_ok=True)

    def get():
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        Path(dest_path).write_bytes(response.content)

    fetch_with_retry(get)
    return dest_path

In [ ]:
def _write_table(df, table_ref, mode="overwrite", merge_schema=False, replace_where=None):
    try:
        writer = df.write.mode(mode).option("overwriteSchema", "true")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if replace_where:
            writer = writer.option("replaceWhere", replace_where)
        writer.saveAsTable(table_ref)
        return True
    except Exception as e:
        print(f"write failed for {table_ref}: {e}")
        return False


def write_bronze(df, table_name, mode="overwrite", replace_where=None):
    return _write_table(df, f"{BRONZE_TABLE}.{table_name}", mode, merge_schema=True, replace_where=replace_where)

In [ ]:
SPANISH_MONTHS = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
]

FORMAT_BY_FILE_TYPE = {"csv": "CSV", "zip": "ZIP", "shapefile": "ZIP"}


def fetch_resources(package_id, base):
    """package_show -> downloadable resources of a dataset."""
    response = requests.get(f"{base}/package_show", params={"id": package_id})
    response.raise_for_status()
    return response.json()["result"]["resources"]


def month_to_description(month):
    """'YYYY-MM' -> 'Month YYYY', the way CKAN spells it in the description."""
    year, month_num = month.split("-")
    return f"{SPANISH_MONTHS[int(month_num) - 1]} {year}"


def select_resource(resources, needle, fmt):
    for resource in resources:
        if needle in resource["description"] and resource["format"] == fmt:
            return resource
    raise ValueError(f"no {fmt} resource for {needle!r}")


def select_by_format(resources, fmt):
    for resource in resources:
        if resource["format"] == fmt:
            return resource
    raise ValueError(f"no {fmt} resource")


def partition_keys(partition, year_start, year_end):
    """Keys to iterate for a dataset: one per year, per month, or a single snapshot."""
    if partition == "snapshot":
        return [None]
    years = range(year_start, year_end + 1)
    if partition == "year":
        return [str(y) for y in years]
    return [f"{y}-{m:02d}" for y in years for m in range(1, 13)]


def pick_resource(resources, partition, key, file_type, select_by):
    fmt = FORMAT_BY_FILE_TYPE[file_type]
    if select_by == "format" or key is None:
        return select_by_format(resources, fmt)
    needle = key if partition == "year" else month_to_description(key)
    return select_resource(resources, needle, fmt)

In [ ]:
def read_csv_spark(path, sep=";"):
    """datos.madrid.es serves both latin-1 and UTF-8; retry on replacement chars."""
    df = spark.read.csv(path, sep=sep, header=True, encoding="UTF-8")
    if any("\ufffd" in c for c in df.columns):
        df = spark.read.csv(path, sep=sep, header=True, encoding="ISO-8859-1")
    return df


def extract_first_file(zip_path, dest_dir):
    with zipfile.ZipFile(zip_path) as z:
        name = z.namelist()[0]
        z.extractall(dest_dir)
    return f"{dest_dir}/{name}"


def read_shapefile(url):
    """Remote shapefile -> DataFrame with geometry as WKT (Spark has no geometry type)."""
    # Deferred import: geopandas
    import geopandas as gpd

    gdf = gpd.read_file(f"zip+{url}").to_crs("EPSG:4326")
    pdf = pd.DataFrame(gdf.assign(geometry=gdf.geometry.to_wkt()))
    return spark.createDataFrame(pdf)


def read_resource(resource, file_type, dataset_destino, key):
    if file_type == "shapefile":
        return read_shapefile(resource["url"])

    landing = f"{LANDING_VOLUME}/{dataset_destino}"
    local = download_to_volume(
        resource["url"], f"{landing}/{key or 'latest'}.{file_type}"
    )
    if file_type == "zip":
        local = extract_first_file(local, landing)
    return read_csv_spark(local)

In [ ]:
def write_silver(df, table_name, mode="overwrite"):
    return _write_table(df, f"{SILVER_TABLE}.{table_name}", mode)

In [ ]:
def write_gold(df, table_name, mode="overwrite"):
    return _write_table(df, f"{GOLD_TABLE}.{table_name}", mode)

In [ ]:
def write_ml(df, table_name, mode="overwrite", replace_where=None):
    return _write_table(df, f"{ML_TABLE}.{table_name}", mode, replace_where=replace_where)

In [ ]:
def rename_columns(df, mapping):
    for old_name, new_name in mapping.items():
        df = df.withColumnRenamed(old_name, new_name)
    return df

In [ ]:
def sanitize_column_names(df):
    """Force lowercase snake_case; source columns are referenced lowercase downstream."""
    for old_name in df.columns:
        new_name = re.sub(r"[^0-9a-zA-Z_]", "_", old_name.strip())
        new_name = re.sub(r"_+", "_", new_name).strip("_").lower()
        if new_name != old_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df

In [ ]:
SPARK_TYPE_MAP = {
    "string": StringType(), "int": IntegerType(), "long": LongType(),
    "short": ShortType(), "byte": ByteType(), "double": DoubleType(),
    "float": FloatType(), "boolean": BooleanType(), "date": DateType(),
    "timestamp": TimestampType(),
}


def resolve_spark_type(data_type):
    data_type = data_type.strip().lower()
    decimal_match = re.match(r"decimal\((\d+),\s*(\d+)\)", data_type)
    if decimal_match:
        precision, scale = decimal_match.groups()
        return DecimalType(int(precision), int(scale))
    return SPARK_TYPE_MAP.get(data_type, StringType())


def apply_schema(df, dataset_destino):
    """Cast per the `column_name,data_type` contract in the bronze Volume, if present."""
    schema_file = f"{BRONZE_SCHEMA_PATH}/{dataset_destino}.csv"
    if not Path(schema_file).exists():
        print(f"apply_schema: no contract for {dataset_destino}, left as is")
        return df
    schema_df = pd.read_csv(schema_file)
    for _, row in schema_df.iterrows():
        column_name = row["column_name"]
        if column_name in df.columns:
            df = df.withColumn(
                column_name, df[column_name].cast(resolve_spark_type(row["data_type"]))
            )
    return df

In [ ]:
def error_record(notebook_name, exception, status="FAILED"):
    return {
        "timestamp": datetime.now().isoformat(),
        "notebook_name": notebook_name,
        "error_message": f"{type(exception).__name__}: {exception}",
        "status": status,
        "run_id": RUN_ID,
    }


def log_errors(errors):
    """Append failures to the infra Delta table. Never raises: one dataset
    failing must not take down the rest of the batch."""
    if not errors:
        return
    df = spark.createDataFrame(errors, schema=ERROR_LOG_SCHEMA)
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{INFRA_TABLE}.error_logs")
    print(f"{len(errors)} errors written to {INFRA_TABLE}.error_logs")

In [ ]:
def log_ingestion(records):
    """Row counts per dataset/partition, queryable with SQL."""
    if not records:
        return
    df = spark.createDataFrame(records, schema=INGESTION_LOG_SCHEMA).withColumn("run_id", lit(RUN_ID))
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{INFRA_TABLE}.ingestion_log")
    print(f"{len(records)} records written to {INFRA_TABLE}.ingestion_log")